In [1]:
import kagglehub, os, torch
dataset_path = kagglehub.dataset_download("sunilthite/ovarian-cancer-classification-dataset")
print("Path to dataset files:", dataset_path)

for dirname, _, filenames in os.walk(dataset_path):
    for filename in filenames[:1]:
        print(os.path.join(dirname, filename))
        

train_dir = os.path.join(dataset_path, "Train_Images")
test_dir = os.path.join(dataset_path, "Test_Images")

/home/conite/anaconda3/envs/GPU_ENV/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /home/conite/.cache/kagglehub/datasets/sunilthite/ovarian-cancer-classification-dataset/versions/1
/home/conite/.cache/kagglehub/datasets/sunilthite/ovarian-cancer-classification-dataset/versions/1/mobilenet_model_224x224_30_10.h5
/home/conite/.cache/kagglehub/datasets/sunilthite/ovarian-cancer-classification-dataset/versions/1/Train_Images/LGSC/18690.png
/home/conite/.cache/kagglehub/datasets/sunilthite/ovarian-cancer-classification-dataset/versions/1/Train_Images/HGSC/3729.png
/home/conite/.cache/kagglehub/datasets/sunilthite/ovarian-cancer-classification-dataset/versions/1/Train_Images/EC/23912.png
/home/conite/.cache/kagglehub/datasets/sunilthite/ovarian-cancer-classification-dataset/versions/1/Train_Images/CC/4697.png
/home/conite/.cache/kagglehub/datasets/sunilthite/ovarian-cancer-classification-dataset/versions/1/Train_Images/MC/7491.png
/home/conite/.cache/kagglehub/datasets/sunilthite/ovarian-cancer-classification-dataset/versions/1/Test_Images/LGSC/1697

In [2]:
import sys
import os
sys.path.append(os.path.abspath('../'))

from src.models.single_network import SingleNetwork
from src.models.mc_dropout import MCDropout
from src.models.deep_ensemble import DeepEnsemble
from src.training.trainer import ModelTrainer
from src.processing.preprocess import get_dataloaders_for_ovarian, get_transforms
from src.processing.loading import load_image_folder

In [3]:
train_transforms = get_transforms(augment=True)
test_transforms = get_transforms()
data_set_train, idx_to_class_train, class_to_idx_train = load_image_folder(train_dir, train_transforms)
data_set_test, idx_to_class_test, class_to_idx_test = load_image_folder(test_dir, test_transforms)

train_loader, val_loader, cal_loader, test_loader = get_dataloaders_for_ovarian(
    dataset_dict={'train': data_set_train, 'test': data_set_test},
    batch_size=64,
    val_split=0.03,
    cal_split=0.12,
)

In [4]:
len(class_to_idx_train)

5

In [5]:
single_net = SingleNetwork(
    num_classes=len(class_to_idx_train),
    learning_rate=5e-5,
    patience=5,
    # device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    device=torch.device('cpu'),
    train_loader=train_loader,
    val_loader=val_loader,
    cal_loader=cal_loader,
    test_loader=test_loader,
    dataset='ovarian',
)
# single_net.train(epochs=50, model_path='single_net_resnet_sipakmed.pth')
single_net.load('../experiments/results/ovarian/best_model_ovarian_resnet_single.pth')

/home/conite/anaconda3/envs/GPU_ENV/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/conite/anaconda3/envs/GPU_ENV/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/home/conite/anaconda3/envs/GPU_ENV/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
/home/conite/Documents/STAGE/HybridUncertaintyDLFramework/src/models/base_model.py:27: FutureWarning: You are us

In [6]:
single_net.model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [7]:
single_net.evaluate()

y_true shape: (3082,) unique: [0 1 2 3 4]
y_proba shape: (3082, 5)
Nombre de classes dans y_true : 5


{'accuracy': 0.7738481505515898,
 'f1_score': 0.7746650220719462,
 'precision': 0.7803077318038397,
 'recall': 0.7738481505515898,
 'roc_auc': 0.9476449222909293,
 'brier_score': 0.32312784,
 'entropy': array([3.9224979e-04, 4.3476682e-02, 1.7830234e-02, ..., 7.1299839e-01,
        1.4994906e-01, 5.9559450e-02], dtype=float32),
 'uncertainty': array([0.39998353, 0.39680687, 0.39890453, ..., 0.24899223, 0.38305625,
        0.3947136 ], dtype=float32),
 'ece': 0.07330981952891882,
 'variance': array([ 13.7156105 ,   1.0799084 ,   1.2496471 ,   8.256605  ,
          4.2424326 ,   9.385279  ,   0.6352446 ,   1.8106363 ,
          5.381059  ,  12.150859  ,  39.765114  ,   1.6938341 ,
          4.0501394 ,  17.38821   ,  15.515826  ,  37.055767  ,
          1.1794944 ,   2.8128607 ,  11.440686  ,  16.098923  ,
         66.38278   ,   0.7946491 ,   4.343563  ,  24.636374  ,
         25.453985  ,  14.983293  ,   0.91934806,   1.1974742 ,
          2.9322596 ,   6.074549  ,  37.76294   ,   1.07

In [8]:
mcdropout = MCDropout(
    num_classes=5,
    learning_rate=5e-5,
    patience=5,
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    train_loader=train_loader,
    val_loader=val_loader,
    cal_loader=cal_loader,
    test_loader=test_loader,
    
)

mcdropout.load('../experiments/results/ovarian/output_mcdropout_resnet50_ovarian.pth')
mcdropout.model

/home/conite/anaconda3/envs/GPU_ENV/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/conite/anaconda3/envs/GPU_ENV/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/home/conite/anaconda3/envs/GPU_ENV/lib/python3.9/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
/home/conite/Documents/STAGE/HybridUncertaintyDLFramework/src/models/base_model.py:27: FutureWarning: You are us

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [9]:
mcdropout.evaluate()

Predictions, leave=False: 100%|██████████| 49/49 [19:17<00:00, 23.62s/it]

y_true shape: (3082,) unique: [0 1 2 3 4]
y_proba shape: (3082, 5)
Nombre de classes dans y_true : 5


{'accuracy': 0.7482154445165476,
 'f1_score': 0.7499614290423345,
 'precision': 0.7607939082372005,
 'recall': 0.7482154445165476,
 'roc_auc': 0.9398170669976746,
 'brier_score': 0.36887184,
 'entropy': array([0.0008678 , 0.06044033, 0.12729247, ..., 0.04497644, 0.11356071,
        0.38118225], dtype=float32),
 'uncertainty': array([0.39996082, 0.39506793, 0.38863993, ..., 0.39659968, 0.38830185,
        0.35665435], dtype=float32),
 'ece': 0.09636233751840362,
 'variance': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 'y_preds': array([0, 0, 0, ..., 4, 4, 4]),
 'y_probs': array([[9.99921620e-01, 6.20377614e-05, 1.46338407e-05, 1.11037389e-06,
         7.27843428e-07],
        [9.90112782e-01, 8.21174029e-03, 1.55976636e-03, 1.17727068e-05,
         1.03912353e-04],
        [9.77211833e-01, 1.44928563e-02, 7.12094596e-03, 7.07218060e-05,
         1.10363867e-03],
        ...,
        [1.04802707e-03, 5.48123941e-03, 7.29270905e-05, 2.08402605e-04,
         9.93189454e-01],
    